In [ ]:
import cv2 as cv
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

In [ ]:
def layer(in_channels,out_channels,kernel_size,padding,batchnorm=True,stride=(1,1)):
    if batchnorm == True:
        sub_model=nn.Sequential(
            nn.Conv2d(in_channels=in_channels,
                    out_channels=out_channels,
                    kernel_size=kernel_size,
                    padding=padding,
                    stride=stride),
                    nn.BatchNorm2d(out_channels),
                    nn.ReLU()
        )
        return sub_model
    else:
        sub_model=nn.Sequential(
            nn.Conv2d(in_channels=in_channels,
                    out_channels=out_channels,
                    kernel_size=kernel_size,
                    padding=padding,
                    stride=stride),
                    nn.ReLU()
        )
        return sub_model


In [ ]:
model=nn.Sequential(
    layer(in_channels=1,out_channels=16,
          kernel_size=(3,3),
          padding=(1,1),batchnorm=True),

    layer(in_channels=16,out_channels=16,
          kernel_size=(3,3),
          padding=(1,1),batchnorm=True),

    nn.MaxPool2d(kernel_size=(2,2),stride=(2,2)),

    nn.Dropout(0.2),

    layer(in_channels=16,out_channels=32,
          kernel_size=(3,3),
          padding=(1,1),batchnorm=True),


      nn.MaxPool2d(kernel_size=(2,2),stride=(2,2)),


      layer(in_channels=32,out_channels=64,
          kernel_size=(3,3),
          padding=(1,1),batchnorm=True),   


    nn.MaxPool2d(kernel_size=(2,2),stride=(2,2)),


    layer(in_channels=64,out_channels=64,
          kernel_size=(3,3),
          padding=(1,1),batchnorm=True),


      nn.Dropout(0.2),

    nn.AdaptiveAvgPool2d(1),

    nn.Flatten(),

    nn.Linear(64,10)
)

In [ ]:
ft_model=nn.Sequential(
    layer(in_channels=1,out_channels=16,
          kernel_size=(3,3),
          padding=(1,1),batchnorm=True),

    layer(in_channels=16,out_channels=16,
          kernel_size=(3,3),
          padding=(1,1),batchnorm=True),

    nn.MaxPool2d(kernel_size=(2,2),stride=(2,2)),

    nn.Dropout(0.2),

    layer(in_channels=16,out_channels=32,
          kernel_size=(3,3),
          padding=(1,1),batchnorm=True),


      nn.MaxPool2d(kernel_size=(2,2),stride=(2,2)),


      layer(in_channels=32,out_channels=64,
          kernel_size=(3,3),
          padding=(1,1),batchnorm=True),   


    nn.MaxPool2d(kernel_size=(2,2),stride=(2,2)),


    layer(in_channels=64,out_channels=64,
          kernel_size=(3,3),
          padding=(1,1),batchnorm=True),


      nn.Dropout(0.2),

    nn.AdaptiveAvgPool2d(1),

    nn.Flatten(),

    nn.Linear(64,7)
)

In [ ]:
if torch.cuda.is_available():
    device='cuda'
else:
    device='cpu'
device

In [ ]:
def num_predict(model, root, device=device):
    model.load_state_dict(torch.load('Weights/Number_Predict_Finall.pth'))
    model.to(device)

    img=cv.imread(root)
    img=cv.cvtColor(img,cv.COLOR_BGR2GRAY)
    df=pd.DataFrame(img)

    start=0
    stop=1
    answer=''

    for i in range(12):
        li=[]

        for i2 in range(28*start,28*stop):
            li.append(df[i2].values)
            x=pd.DataFrame(li).transpose()

        for i3 in range(1):
            model.eval()
            x=x.to_numpy()
            x=torch.tensor(x,dtype=torch.float32).to(device)
            x=x.unsqueeze(0)
            x=x.unsqueeze(0)
            pred=model(x)
            sft=nn.Softmax(dim=1)
            f=sft(pred).argmax()
            answer+=str(f.item())
            
        start+=1
        stop+=1
        
    return answer


In [ ]:
label_to_idx={
        'division':0,
        'addition':1,
        'equality':2,
        'subtraction':3,
        'X':4,
        'null':5,
        'number':6}

In [ ]:
def operator_predict(ft_model, predict, root,label_to_idx ,device=device):
    ft_model.load_state_dict(torch.load('Weights/Finetune_Finall.pth'))
    ft_model.to(device)

    img=cv.imread(root)
    img=cv.cvtColor(img,cv.COLOR_BGR2GRAY)
    df=pd.DataFrame(img)

    start=0
    stop=1
    answer=list(predict)

    for i in range(12):
        li=[]

        for i2 in range(28*start,28*stop):
            li.append(df[i2].values)
        x=pd.DataFrame(li).transpose()

        for i3 in range(1):
            ft_model.eval()
            x=x.to_numpy()
            x=torch.tensor(x,dtype=torch.float32).to(device)
            x=x.unsqueeze(0)
            x=x.unsqueeze(0)
            pred=ft_model(x)
            sft=nn.Softmax(dim=1)
            f=sft(pred).argmax()

            for class_name in label_to_idx:
                if label_to_idx.get(class_name)==f:

                    if f==0:
                        answer.pop(i)
                        answer.insert(i,'/')

                    elif f==1:
                        answer.pop(i)
                        answer.insert(i,'+')

                    elif f==2:
                        answer.pop(i)
                        answer.insert(i,'=')

                    elif f==3:
                        answer.pop(i)
                        answer.insert(i,'-')

                    elif f==4:
                        answer.pop(i)
                        answer.insert(i,'*') 

                    elif f==5:
                        answer.pop(i)
                        answer.insert(i,' ')
                        
                    elif f==6:
                        continue
                    
        start+=1
        stop+=1

    final_predict=''
    for string in answer:
        final_predict+=string


    return final_predict , root

In [ ]:
#Address Tasvir Az Folder Final_Test_Images
root_test=r'Final_Test_Images/10.jpg'  

In [ ]:
#Pishbini Model
predict=num_predict(model, root_test)
final_predict,root=operator_predict(ft_model, predict, root_test, label_to_idx)
final_predict

In [ ]:
#Tasvir Asli
imagee=cv.imread(root_test)
imagee=cv.cvtColor(imagee,cv.COLOR_BGR2GRAY)
plt.imshow(imagee,cmap='gray')